In [0]:
# Databricks notebook
# MAGIC %md
# MAGIC # 01 - Bronze: endpoint leggeri OpenF1 (incrementale)
# MAGIC - `sessions` e `meetings`: ricaricati per anno (`replaceWhere` su `year`).
# MAGIC - Endpoint per sessione: ricaricati per `_session_key` (`replaceWhere`, atomico).
# MAGIC - Incrementale: salta le coppie (sessione, endpoint) gia' nel log `brz._ingestion_log`,
# MAGIC   ma ricarica le sessioni finite da meno di `refresh_days` giorni (dati ancora incompleti).
# MAGIC - Parte dalla sessione piu' recente. `max_sessions` limita quante sessioni elaborare (utile per i test).
# MAGIC - In bronze tutti i campi sono stringhe (nested come JSON): la tipizzazione si fa in silver.

import json
import time
import requests
from urllib.parse import urlencode
from datetime import datetime, timedelta, timezone
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

dbutils.widgets.text("year", "2025")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "full"])
dbutils.widgets.text("refresh_days", "3")
dbutils.widgets.text("max_sessions", "")   # vuoto = tutte

YEAR = int(dbutils.widgets.get("year"))
CATALOG = "f1"
MODE = dbutils.widgets.get("mode")
REFRESH_DAYS = int(dbutils.widgets.get("refresh_days"))
MAX_SESSIONS = int(dbutils.widgets.get("max_sessions")) if dbutils.widgets.get("max_sessions").strip() else None

BASE = "https://api.openf1.org/v1"
LOG_TABLE = f"{CATALOG}.cfg.ingestion_log"

# Endpoint scaricabili per session_key (car_data e location sono nel notebook 02)
SESSION_ENDPOINTS = ["drivers", "laps", "stints", "pit", "position",
                     "intervals", "weather", "race_control"]



In [0]:
def get_json(path, params=None):
    """GET con retry; 404 = nessun dato."""
    url = f"{BASE}/{path}?" + urlencode(params or {})
    for attempt in range(6):
        try:
            resp = requests.get(url, timeout=120)
            if resp.status_code == 429:
                time.sleep(2 * (attempt + 1))
                continue
            if resp.status_code == 404:
                return []
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"Richiesta fallita: {url}")


def to_str(v):
    if v is None:
        return None
    if isinstance(v, (list, dict)):
        return json.dumps(v)
    return str(v)


def write_bronze(endpoint, rows, predicate, session_key=None):
    """Sostituisce in modo atomico le righe che rispettano `predicate` (replaceWhere)."""
    if not rows:
        return 0
    table = f"{CATALOG}.brz.{endpoint}"
    ingested = datetime.now(timezone.utc)
    cols = sorted({k for r in rows for k in r})
    schema = StructType(
        [StructField(c, StringType(), True) for c in cols]
        + [StructField("_session_key", StringType(), True),
           StructField("_ingested_at", TimestampType(), True)]
    )
    sk = str(session_key) if session_key is not None else None
    records = [tuple(to_str(r.get(c)) for c in cols) + (sk, ingested) for r in rows]
    df = spark.createDataFrame(records, schema)

    writer = df.write.format("delta").option("mergeSchema", "true")
    if spark.catalog.tableExists(table):
        writer.mode("overwrite").option("replaceWhere", predicate).saveAsTable(table)
    else:
        writer.mode("overwrite").saveAsTable(table)
    return len(rows)


def log_load(session_key, endpoint, n):
    schema = StructType([
        StructField("session_key", StringType()), StructField("endpoint", StringType()),
        StructField("row_count", LongType()), StructField("ingested_at", TimestampType())])
    spark.createDataFrame([(str(session_key), endpoint, n, datetime.now(timezone.utc))], schema) \
         .write.format("delta").mode("append").saveAsTable(LOG_TABLE)

In [0]:
# Tabelle per anno: sessions e meetings (sempre ricaricate: sono piccole)
for endpoint in ["sessions", "meetings"]:
    rows = get_json(endpoint, {"year": YEAR})
    n = write_bronze(endpoint, rows, predicate=f"year = '{YEAR}'")
    print(f"{endpoint}: {n} righe")
    time.sleep(0.5)


In [0]:
# Sessioni gia' iniziate, dalla piu' recente
now = datetime.now(timezone.utc)
sessions = []
for s in spark.sql(
        f"SELECT session_key, date_start, date_end FROM {CATALOG}.brz.sessions "
        f"WHERE year = '{YEAR}'").collect():
    if not s.date_start:
        continue
    start = datetime.fromisoformat(s.date_start)
    if start > now:
        continue  # sessione futura
    end = datetime.fromisoformat(s.date_end) if s.date_end else start
    sessions.append((int(s.session_key), start, end))
sessions.sort(key=lambda x: x[1], reverse=True)

# Cosa e' gia' stato caricato
done = {(r.session_key, r.endpoint) for r in
        spark.sql(f"SELECT DISTINCT session_key, endpoint FROM {LOG_TABLE}").collect()}
refresh_from = now - timedelta(days=REFRESH_DAYS)

# Lavoro da fare: (sessione -> endpoint da caricare)
todo = {}
for sk, start, end in sessions:
    recent = end >= refresh_from
    eps = [ep for ep in SESSION_ENDPOINTS
           if MODE == "full" or recent or (str(sk), ep) not in done]
    if eps:
        todo[sk] = eps

todo_keys = list(todo)[:MAX_SESSIONS] if MAX_SESSIONS else list(todo)
print(f"{len(sessions)} sessioni iniziate nel {YEAR}, {len(todo)} da caricare, "
      f"{len(todo_keys)} in questa esecuzione")

In [0]:
for sk in todo_keys:
    for endpoint in todo[sk]:
        rows = get_json(endpoint, {"session_key": sk})
        n = write_bronze(endpoint, rows, predicate=f"_session_key = '{sk}'", session_key=sk)
        log_load(sk, endpoint, n)   # registrato anche con 0 righe (es. intervals nelle prove)
        print(f"session {sk} | {endpoint}: {n} righe")
        time.sleep(0.4)  # resta sotto il rate limit

In [0]:
# Controllo finale: righe per tabella
tables = ["sessions", "meetings"] + SESSION_ENDPOINTS
summary = [(t, spark.table(f"{CATALOG}.brz.{t}").count())
           for t in tables if spark.catalog.tableExists(f"{CATALOG}.brz.{t}")]
display(spark.createDataFrame(summary, ["tabella", "righe"]))